In [12]:
import sys
from pathlib import Path

# --- Auto-detect project root (works from repo root or notebooks/) ---
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_RAW_DIR / "Divar.csv"
OUTPUT_PATH = DATA_PROCESSED_DIR / "preprocessing_output_v1.parquet"

print(f"Project root : {BASE_DIR}")
print(f"Input path   : {INPUT_PATH}")
print(f"Output path  : {OUTPUT_PATH}")


Project root : c:\Users\98936\Desktop\divar-housing-analysis
Input path   : c:\Users\98936\Desktop\divar-housing-analysis\data\raw\Divar.csv
Output path  : c:\Users\98936\Desktop\divar-housing-analysis\data\processed\preprocessing_output_v1.parquet


In [13]:
import os
import pandas as pd

# --- Make the project root importable so `source.` package resolves ---
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from source.financial_cleaning import clean_financial_pipeline
from source.financial_feature_engineering import feature_engineering_pipeline

In [14]:
assert INPUT_PATH.exists(), f"Input file not found at: {INPUT_PATH}"

# --- Load (extension-aware, so a future Parquet input won't crash it) ---
print(f"Loading data from: {INPUT_PATH}")
if INPUT_PATH.suffix == ".parquet":
    df_raw = pd.read_parquet(INPUT_PATH)
else:
    df_raw = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Raw data shape: {df_raw.shape}")

# --- Step 1: Cleaning (winsorize=False -> no leakage from future test set) ---
print("\n--- Step 1: Financial Cleaning ---")
df_step1, df_daily = clean_financial_pipeline(df_raw, verbose=True, winsorize=False)

# --- Step 2: Feature Engineering ---
print("\n--- Step 2: Feature Engineering ---")
df_final = feature_engineering_pipeline(df_step1, verbose=True)

print(f"\nPreprocessing completed. Final shape: {df_final.shape}")
print(f"Daily-rent subset kept aside: {df_daily.shape}")

Loading data from: c:\Users\98936\Desktop\divar-housing-analysis\data\raw\Divar.csv
Raw data shape: (1000000, 61)

--- Step 1: Financial Cleaning ---
[Data Cleaning] Impossible values mapped to NaN:
  - price_value: 13,186 rows
  - credit_value: 3,109 rows
  - rent_value: 489 rows
[Data Cleaning] Dropping dead/redundant columns: ['rent_type', 'transformable_price', 'transformable_credit', 'transformable_rent', 'transformed_credit', 'transformed_rent']

--- Step 2: Feature Engineering ---
[Financial Feature Engineering] Successfully added features: ['equivalent_full_credit', 'weekend_price_ratio', 'special_day_price_ratio', 'is_rent_credit_convertible', 'allows_single_tenant', 'log_price_value', 'log_rent_value', 'log_credit_value', 'log_equivalent_full_credit']

Preprocessing completed. Final shape: (1000000, 63)
Daily-rent subset kept aside: (18068, 66)


### Column Count Summary

| Stage | Change | Count |
|---|---|---|
| Raw input | — | 61 |
| Dropped dead columns | `rent_type`, `transformable_price`, `transformable_credit`, `transformable_rent`, `transformed_credit`, `transformed_rent` | −6 |
| Added in cleaning | `deal_type`, `is_negotiable`, `is_transformable`, `is_full_mortgage`, `equivalent_full_credit` | +5 |
| Added in feature engineering | `weekend_price_ratio`, `special_day_price_ratio`, `is_rent_credit_convertible`, `allows_single_tenant` | +4 |
| Log transforms | `log_price_value`, `log_credit_value`, `log_rent_value`, `log_equivalent_full_credit` | +4 |
| Dropped intermediate sources | `rent_price_on_regular_days`, `rent_price_at_weekends`, `rent_price_on_special_days`, `rent_to_single`, `rent_credit_transform` | −5 |
| **Final** | 61 − 6 + 5 + 4 + 4 − 5 | **63** |

> Note: `winsorize=False` in this notebook. Capping must be computed on the training split only (see `06_ml_prep`) to avoid data leakage.


In [15]:
def export_final_parquet(df: pd.DataFrame, parquet_path: Path) -> None:
    """Persist the processed dataset as Parquet with PyArrow-friendly dtypes."""
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    df_to_save = df.copy()

    # Cast text columns (object AND already-string) so PyArrow serializes cleanly
    text_cols = df_to_save.select_dtypes(include=["object", "string"]).columns
    if len(text_cols) > 0:
        df_to_save[text_cols] = df_to_save[text_cols].astype("string")

    df_to_save.to_parquet(parquet_path, index=False, engine="pyarrow")

    size_mb = os.path.getsize(parquet_path) / (1024 * 1024)
    print(f"Saved Parquet to: {parquet_path}")
    print(f"File size       : {size_mb:.2f} MB")


export_final_parquet(df_final, OUTPUT_PATH)

print("\n--- Final Pipeline Report ---")
print(f"Total rows   : {df_final.shape[0]:,}")
print(f"Total columns: {df_final.shape[1]}")

Saved Parquet to: c:\Users\98936\Desktop\divar-housing-analysis\data\processed\preprocessing_output_v1.parquet
File size       : 295.04 MB

--- Final Pipeline Report ---
Total rows   : 1,000,000
Total columns: 63
